# 추론 모델 밑바닥부터 4편 실습 노트북 — 5장: 추론 시간 확장 ② 점수 매기기와 스스로 고치기 (한국어 주석판)

**출처:** 원본 노트북 `ch05_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch05/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** 스스로 고치기는 답을 여러 번 다시 쓰기 때문에 오래 걸립니다. 처음에는 적은 문항으로 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 5장: 스스로 고치기로 추론 시간 확장하기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "matplotlib",
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F01_raschka.webp" width=600>

&nbsp;
## 5.1 모델의 답에 점수를 매기고 거듭 고치기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F02_raschka.webp" width=600>

- 이 절에는 코드가 없습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F03_raschka.webp" width=600>

&nbsp;
## 5.2 사전학습된 모델 불러오기

- 모델을 불러오는 코드는 2~4장과 비슷합니다.

- 아래에서는 장치를 `"cpu"`로 못 박아 둡니다. 컴퓨터나 운영체제가 달라도 결과가 최대한 비슷하게 나오게 하려는 것입니다.
- 먼저 이 장의 코드를 그대로 한 번 돌려 보세요. 그다음에 아래의 `device = torch.device("cpu")` 줄을 지우는 것을 권합니다.
- `"cpu"`로 고정해도 결과가 여러분 컴퓨터에서는 조금 다를 수 있습니다. 이 장에서는 아주 작은 숫자를 많이 곱하기 때문입니다(특히 5.4절부터 그렇습니다).

In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
# 결과를 맞추려고 CPU로 고정합니다 (지우면 GPU를 씁니다)
device = torch.device("cpu")  # optionally delete to allow running on GPU

# 기본(base) 모델과 토크나이저를 불러옵니다
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 앞 장에서 다룬 MATH-500 데이터셋의 문제 하나로 모델을 시험해 봅시다.

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
# 문제를 모델이 읽는 프롬프트 형식으로 감쌉니다
prompt = render_prompt(raw_prompt)
# "단계별로 설명하라"를 붙여 풀이 과정을 끌어냅니다
prompt_cot = prompt + "\n\nExplain step by step."

# 시드를 고정해 같은 답이 다시 나오게 합니다
torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

In [ ]:
# 시드만 바꿔 두 번째 답을 만듭니다
torch.manual_seed(3)
response_2 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9, 
)

In [ ]:
# 두 답의 길이를 글자 수와 토큰 수로 비교합니다
print("Response 1 characters:", len(response_1))
print("Response 1 tokens:", len(tokenizer.encode(response_1)))
print("\nResponse 2 characters:", len(response_2))
print("Response 2 tokens:", len(tokenizer.encode(response_2)))

- 참고로 정답은 83입니다.

&nbsp;
## 5.3 규칙 기반 점수로 LLM의 답 채점하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F04_raschka.webp" width=600>

- 자기 일관성(self-consistency)에서 표가 동점일 때 승부를 가르는 데 쓸 수 있습니다.
- 자기 일관성으로 얻은 답 전부에 점수를 매겨 순위를 정하는 데도 쓸 수 있습니다(이것을 Best-of-N 추론 확장이라고 부릅니다).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F05_raschka.webp" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate
import math

def heuristic_score(
    answer,
    prompt=None,  # Placeholder that is ignored
    brevity_bonus=500.0,
    boxed_bonus=2.0,
    extract_bonus=1.0,
    fulltext_bonus=0.0,
):
    # 점수는 0에서 시작해 조건마다 더해 갑니다
    score = 0.0

    # Reward answers that have a final boxed value
    cand = extract_final_candidate(answer, fallback=None)
    if cand:
        score += boxed_bonus

    # Give weaker rewards if answer doesn't have a boxed value
    else:
        cand = extract_final_candidate(answer, fallback="number_only")
        if cand:
            score += extract_bonus
        else:
            cand = extract_final_candidate(
                answer, fallback="number_then_full"
            )
            if cand:
                score += fulltext_bonus

    # Add a brevity reward that decays with text length
    # 짧을수록 최대 1.5점에 가까운 보너스를 받습니다
    score += 1.5 * math.exp(-len(answer) / brevity_bonus)
    return score

- 이 코드는 단순하게 하려고 토큰 수 대신 글자 수를 셉니다. 그래서 여기에 토크나이저를 넘길 필요가 없습니다. 물론 토큰 수로 쉽게 바꿀 수도 있습니다.

In [ ]:
import matplotlib.pyplot as plt

def plot_brevity_curve(brevity_bonus, max_len=2048):
    lengths = torch.arange(1, max_len)
    # heuristic_score의 짧음 보너스와 같은 식입니다
    scores = 1.5 * torch.exp(-lengths / brevity_bonus)

    plt.figure(figsize=(4, 3))
    plt.plot(lengths, scores)
    plt.xlabel("Text length (number of characters)")
    plt.ylabel("Score contribution")
    plt.tight_layout()
    #plt.savefig("brevity_curve.pdf")
    plt.show()

# brevity_bonus=500일 때 곡선을 그립니다
plot_brevity_curve(500)

In [ ]:
print(round(heuristic_score(response_1), 3))

In [ ]:
print(round(heuristic_score(response_2), 3))

&nbsp;
## 5.4 토큰 확률 점수 이해하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F07_raschka.webp" width=600>

- 토큰 가능도(token likelihood)라고도 부릅니다.
- 생성한 글에 점수를 매기는 함수를 만들어 봅니다.
- 이 함수는 곧 나올 스스로 고치기(self-refinement) 방법에서 씁니다. `heuristic_score`처럼 자기 일관성 방법에서 동점을 가르는 데 써도 됩니다.
- 효율만 따지면 4.5절의 `generate_text_top_p_stream_cache` 함수 안에서 점수를 같이 계산할 수도 있습니다. 하지만 여기서는 따로 떼어 둡니다. 그래야 `heuristic_score`처럼 갈아 끼울 수 있는 채점 함수가 됩니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F08_raschka.webp" width=600>

- 위 그림은 앞 장에서 본 것으로, 다음 토큰이 어떻게 만들어지는지 보여 줍니다.
- 여기서는 다음 토큰을 실제로 만들지 않습니다. 다음 토큰 후보들의 점수만 궁금합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F09_raschka.webp" width=600>

- 위 그림은 단순하게 로짓(logit) 점수로 보여 주었습니다. 하지만 확률(토큰 가능도)로 보면 더 알아보기 쉽습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F10_raschka.webp" width=600>

- 확률 점수에는 문제가 있습니다. 대부분이 0에 가깝다는 점입니다(다음 절에서 해결합니다).
- LLM이 "Germany's capital is Berlin"이라는 답을 만든다고 해 봅시다. 토큰으로 나누면 5개입니다.
- LLM은 각 토큰에 확률을 매깁니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F11_raschka.webp" width=600>

In [ ]:
@torch.inference_mode()
def calc_next_token_probas(model, tokenizer, prompt, device, show=True):

    # 글을 토큰 ID로 바꿉니다
    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    # Get logits and probabilities similar to text generation functions
    logits = model(token_ids.unsqueeze(0)).squeeze(0)
    all_probas = torch.softmax(logits, dim=-1)

    # Positions we score (here: all)
    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)

    # Since we have the text, we know the true next tokens
    next_ids = token_ids[1:]

    # Get probabilities for each next token
    next_token_probas = all_probas[t_idx, next_ids]

    # Likelihood of the sequence is the product of the probability scores
    prod_next_token_probas = torch.prod(next_token_probas)

    # show=True면 출력만 하고, 아니면 값을 돌려줍니다
    if show:
        print("Next-token probabilities:", next_token_probas)
        print("Joint probability:", prod_next_token_probas)

    else:
        return next_token_probas, prod_next_token_probas

\begin{align}
P(x_1, x_2, \dots, x_T \mid W)
&= \prod_{t=1}^{T} P(x_t \mid x_{1:t-1}, W) \\
&= P(x_1 \mid W) \, \cdot \,
   P(x_2 \mid x_1, W) \, \cdot \,
   P(x_3 \mid x_{1:2}, W) \,
   \cdots \,
   P(x_T \mid x_{1:T-1}, W)
\end{align}

- 여기서 $W$는 모델 파라미터(학습된 가중치), $x$는 토큰 시퀀스, $T$는 토큰 수입니다.
- 아래 숫자들은 0에 아주 가까워서 CPU, MPS, CUDA 장치마다 조금씩 다르게 나옵니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F12_raschka.webp" width=600>

In [ ]:
torch.set_printoptions(precision=4, sci_mode=True)

In [ ]:
# 맞는 답(Berlin)의 확률
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

In [ ]:
# 엉뚱한 답(Bridge)의 확률
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

In [ ]:
# 그럴듯하지만 틀린 답(Hamburg)의 확률
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Hamburg"
    )

&nbsp;
## 5.5 토큰 확률 점수에서 로그 확률로

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F13_raschka.webp" width=600>

- 개념을 하나 더 소개합니다. 로그 확률(log-probability, 줄여서 logprob)입니다. 앞 절에서 소프트맥스 함수로 구한 확률과 비슷한데, 거기에 로그를 씌운 것뿐입니다.

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
# -2부터 2까지 7개 로짓을 예시로 만듭니다
logits = torch.linspace(-2, 2, steps=7)
probas = torch.softmax(logits, dim=-1)
print(probas)

In [ ]:
print(torch.log(probas))

In [ ]:
log_probas = torch.log_softmax(logits, dim=-1)
print(log_probas)

In [ ]:
# 로짓·확률·로그 확률을 나란히 세 그래프로 그립니다
plt.figure(figsize=(9, 4))

# Logits
plt.subplot(1, 3, 1)
plt.bar(range(len(logits)), logits, color="C0", alpha=0.7)
plt.title("Logits")
plt.xlabel("Token index")
plt.ylabel("Value")
plt.grid(alpha=0.3)

# Softmax
plt.subplot(1, 3, 2)
plt.bar(range(len(probas)), probas, color="C1", alpha=0.7)
plt.title("torch.softmax(logits)")
plt.xlabel("Token index")
plt.ylabel("Probability")
plt.ylim(0, 1)
plt.grid(alpha=0.3)

# Log-softmax
plt.subplot(1, 3, 3)
plt.bar(range(len(log_probas)), log_probas, color="C2", alpha=0.7)
plt.title("torch.log_softmax(logits)")
plt.xlabel("Token index")
plt.ylabel("Log-probability")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("logits_softmax_log_softmax.pdf")
plt.show()

- 로그 확률은 값의 범위가 적당해서 수치 계산이 안정적입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F15_raschka.webp" width=600>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F16_raschka.webp" width=600>

\begin{align}
\log P(x_1, x_2, \dots, x_T \mid W)
&= \log \prod_{t=1}^{T} P(x_t \mid x_{1:t-1}, W) \\
&= \sum_{t=1}^{T} \log P(x_t \mid x_{1:t-1}, W) \\
&= \log P(x_1 \mid W)
   + \log P(x_2 \mid x_1, W)
   + \log P(x_3 \mid x_{1:2}, W)
   + \cdots
   + \log P(x_T \mid x_{1:T-1}, W).
\end{align}

In [ ]:
@torch.inference_mode()
def calc_next_token_logprobas(model, tokenizer, prompt, device, show=True):

    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    logits = model(token_ids.unsqueeze(0)).squeeze(0)
    # We now use log_softmax
    all_logprobas = torch.log_softmax(logits, dim=-1)

    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)
    next_ids = token_ids[1:]
    # 실제 다음 토큰의 로그 확률만 골라냅니다
    next_token_logprobas = all_logprobas[t_idx, next_ids]

    # We replace the product with a sum
    sum_next_token_logprobas = torch.sum(next_token_logprobas)

    if show:
        print("Next-token log-probabilities:", next_token_logprobas)
        print("Joint log-probability:", sum_next_token_logprobas)
    else:
        return next_token_logprobas, sum_next_token_logprobas

In [ ]:
# 맞는 답(Berlin)의 로그 확률
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

In [ ]:
# 엉뚱한 답(Bridge)의 로그 확률
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

In [ ]:
# 그럴듯하지만 틀린 답(Hamburg)의 로그 확률
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Hamburg"
    )

&nbsp;
## 5.6 로그 확률로 모델의 확신 정도 채점하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F17_raschka.webp" width=600>

- 로그 확률 점수를 휴리스틱 점수처럼 쓸 수 있는 채점 함수로 바꿉니다.
- 앞 절과 달라진 점은 두 가지입니다. (1) 프롬프트는 빼고 답만 채점합니다. (2) 토큰 점수들의 평균을 냅니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F18_raschka.webp" width=600>

In [ ]:
example_prompt = "What is the capital of Germany?"
example_answer = " The capital of Germany is Berlin."

# 질문과 답을 이어 붙여 통째로 채점합니다
next_token_logprobas, sum_next_token_logprobas = calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt=example_prompt+example_answer,
        show=False
    )

print("Next-token logprobas:", next_token_logprobas)
print("Joint log-probability:", sum_next_token_logprobas)

In [ ]:
print(len(tokenizer.encode(example_answer)))

In [ ]:
last_7 = next_token_logprobas[-7:]
print(last_7)

In [ ]:
print(torch.mean(last_7))

In [ ]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps)

In [ ]:
# 맞는 답의 평균 로그 확률
score_1 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Berlin.",
    device=device
)
print(score_1)

In [ ]:
# 틀린 답의 평균 로그 확률
score_2 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Bridge.",
    device=device
)
print(score_2)

In [ ]:
# 5.2절 첫 번째 답의 평균 로그 확률
avg_logprob_answer(
    model, tokenizer,
    prompt=prompt_cot,
    answer=response_1,
    device=device
)

In [ ]:
# 5.2절 두 번째 답의 평균 로그 확률
avg_logprob_answer(
    model, tokenizer,
    prompt=prompt_cot,
    answer=response_2,
    device=device
)

&nbsp;
## 5.7 되풀이 피드백으로 스스로 고치기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F19_raschka.webp" width=600>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F20_raschka.webp" width=600>

In [ ]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(123)
# 고치기 전 첫 답(초안)을 만듭니다
initial_response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

In [ ]:
# 초안을 검토해 달라는 비평 프롬프트를 만듭니다
def make_critique_prompt(raw_prompt, draft):
    return (
        "You are a meticulous reviewer. Identify logical errors, missing "
        "steps, or arithmetic mistakes. If the answer seems correct, "
        "say so briefly. Then propose a concise plan to fix issues.\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Draft answer:\n{draft}\n\n"
        "Write a short critique and bullet-point fix plan "
        "(under ~120 words).\n"
        "Critique:"
    )


critique_prompt = make_critique_prompt(raw_prompt, initial_response)
torch.manual_seed(123)
# 같은 모델이 자기 초안을 비평합니다
critique = generate_text_stream_concat_flex(
    model, tokenizer, critique_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

In [ ]:
# 질문·초안·비평을 넣어 고친 답을 요청하는 프롬프트
def make_refine_prompt(raw_prompt, draft, critique):
    return (
        "Revise the answer using the critique. Keep it concise and "
        "end with a final boxed result: \\boxed{ANSWER}\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Previous answer:\n{draft}\n\n"
        f"Critique:\n{critique}\n\n"
        "Revised answer:"
    )

refine_prompt = make_refine_prompt(raw_prompt, initial_response, critique)
torch.manual_seed(123)
# 비평을 반영해 고친 답을 만듭니다
revised_answer = generate_text_stream_concat_flex(
    model, tokenizer, refine_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

&nbsp;
## 5.8 스스로 고치기 루프 코딩하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F21_raschka.webp" width=600>

In [ ]:
def self_refinement_loop(
    model,
    tokenizer,
    raw_prompt,
    device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=None,
    prompt_renderer=render_prompt,
    prompt_suffix="",
    verbose=False,
    temperature=0.7,
    top_p=0.9,
):
    # 반복마다 기록을 쌓아 둘 목록
    steps = []

    # Initial response (draft)
    prompt = prompt_renderer(raw_prompt) + prompt_suffix
    current_full = generate_text_stream_concat_flex(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=max_response_tokens,
        verbose=False,
        generate_func=generate_text_top_p_stream_cache,
        temperature=temperature,
        top_p=top_p,
    )

    # 초안에서 최종 답만 뽑습니다
    current_extracted = extract_final_candidate(
        current_full, fallback="number_then_full"
    )
    # 채점 함수가 있으면 초안 점수를 매깁니다
    if score_fn:
        current_score = score_fn(answer=current_full, prompt=prompt)
    else:
        current_score = 0.0

    # Run for one or more iterations
    for it in range(iterations):
        draft_before_full = current_full
        draft_before_extracted = current_extracted
        score_before = current_score

        # Critique the response
        critique_prompt = make_critique_prompt(
            raw_prompt, draft_before_full
        )
        critique_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=critique_prompt,
            device=device,
            max_new_tokens=max_critique_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        # Refine the response
        refine_prompt = make_refine_prompt(
            raw_prompt, draft_before_full, critique_full
        )
        revised_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=refine_prompt,
            device=device,
            max_new_tokens=max_response_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        # 고친 답에서도 최종 답을 뽑습니다
        revised_extracted = extract_final_candidate(
            revised_full, fallback="number_then_full"
        )
        if score_fn:
            revised_score = score_fn(
                answer=revised_full, prompt=prompt  # Still use original prompt here
            )
        else:
            revised_score = 0.0
        
        # Log the results
        step = {
            "iteration": it + 1,
            "draft_full": draft_before_full,
            "draft_extracted": draft_before_extracted,
            "critique": critique_full,
            "revised_full": revised_full,
            "revised_extracted": revised_extracted,
            "score_before": score_before,
            "score_after": revised_score,
        }
        steps.append(step)

        if verbose:
            print(
                f"[Refinement {it+1}/{iterations}]"
                f"\nCurrent: {draft_before_extracted}"
                f"\nRevised: {revised_extracted}"
                f"\nScore before: {score_before:.3f}"
                f"\nScore after: {revised_score:.3f}"
                f"\n{'=' * 25}"
            )

        # Accept revised response if it's not worse
        if revised_score >= current_score:
            current_full = revised_full
            current_extracted = revised_extracted
            current_score = revised_score

    return {
        "final_full": current_full,
        "final_extracted": current_extracted,
        "steps": steps,
    }

In [ ]:
from functools import partial

# 모델 등을 미리 채워 score_fn 모양(answer, prompt)으로 맞춥니다
avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)

In [ ]:
torch.manual_seed(1)

# 평균 로그 확률을 점수로 삼아 2번 고쳐 봅니다
results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

In [ ]:
print(results_logprob["final_extracted"])

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F22_raschka.webp" width=600>

&nbsp;
## 5.9 정리

- 이 절에는 코드가 없습니다.